In [265]:
from PIL import Image
import os
import numpy as np
import random
import math

In [266]:
def load_character_templates(character_variation, num_variants=30):
    characters = {}

    punctuations = ".,';:?!\""

    for i in range(26):
        lowercase = chr(ord("a") + i)
        uppercase = chr(ord("A") + i)
        if i < len(punctuations):
            punctuation = punctuations[i]

        if character_variation == "LOW":
            lowercase_paths = [f"lowercase_characters/{lowercase}/0.jpg"]
            uppercase_paths = [f"uppercase_characters/{uppercase}/0.jpg"]
            if i < len(punctuations):
                punctuation_paths = [f"punctuation/{punctuation}/0.jpg"]

        elif character_variation == "HIGH":
            lowercase_paths = [
                f"lowercase_characters/{lowercase}/{j}.jpg" for j in range(num_variants)
            ]
            uppercase_paths = [
                f"uppercase_characters/{uppercase}/{j}.jpg" for j in range(num_variants)
            ]
            if i < len(punctuations):
                punctuation_paths = [
                    f"punctuation/{punctuation}/{j}.jpg" for j in range(num_variants)
                ]

        characters[lowercase] = [
            Image.open(path).convert("L")
            for path in lowercase_paths
            if os.path.exists(path)
        ]

        characters[uppercase] = [
            Image.open(path).convert("L")
            for path in uppercase_paths
            if os.path.exists(path)
        ]

        if i < len(punctuations):
            characters[punctuation] = [
                Image.open(path).convert("L")
                for path in punctuation_paths
                if os.path.exists(path)
            ]

    return characters

In [267]:
def transform_character(img, baseline_anchor, rotation_deg=0.0, vertical_stretch=1.0):
    w, h = img.size
    new_h = max(1, int(round(h * vertical_stretch)))
    stretched = img.resize((w, new_h), Image.Resampling.BICUBIC)

    stretched_baseline = baseline_anchor * vertical_stretch

    if abs(rotation_deg) < 1e-8:
        return stretched, stretched_baseline

    rotated = stretched.rotate(rotation_deg, expand=True, fillcolor=255)

    w1, h1 = stretched.size
    cx, cy = w1 / 2.0, h1 / 2.0
    px, py = cx, stretched_baseline

    theta = math.radians(rotation_deg)
    cos_t = math.cos(theta)
    sin_t = math.sin(theta)

    def rotate_point(x, y):
        x0 = x - cx
        y0 = y - cy
        xr = cos_t * x0 - sin_t * y0 + cx
        yr = sin_t * x0 + cos_t * y0 + cy
        return xr, yr

    corners = [
        rotate_point(0, 0),
        rotate_point(w1, 0),
        rotate_point(0, h1),
        rotate_point(w1, h1),
    ]

    min_x = min(x for x, y in corners)
    min_y = min(y for x, y in corners)

    prx, pry = rotate_point(px, py)

    new_anchor_y = pry - min_y

    return rotated, new_anchor_y

In [268]:
STRUCTURE_PRESETS = {
    "LOW": {
        "base_letter_spacing": 6,
        "base_word_spacing": 40,
        "baseline_jitter": 0,
        "char_spacing_jitter": 0,
        "word_spacing_jitter": 0,
        "rotation_range": (0, 0),
        "vertical_stretch_range": (1.0, 1.0),
    },
    "HIGH": {
        "base_letter_spacing": 6,
        "base_word_spacing": 40,
        "baseline_jitter": 15,
        "char_spacing_jitter": 6,
        "word_spacing_jitter": 15,
        "rotation_range": (-10, 10),
        "vertical_stretch_range": (0.9, 1.15),
    }
}

def sample_structure_values(
    base_letter_spacing,
    base_word_spacing,
    baseline_jitter=0,
    char_spacing_jitter=0,
    word_spacing_jitter=0,
    rotation_range=(0.0, 0.0),
    vertical_stretch_range=(1.0, 1.0)
):
    baseline_offset = random.randint(-baseline_jitter, baseline_jitter)

    char_spacing = base_letter_spacing + random.randint(
        -char_spacing_jitter, char_spacing_jitter
    )

    char_spacing = max(0, char_spacing)

    rotation_deg = random.uniform(rotation_range[0], rotation_range[1])

    vertical_stretch = random.uniform(
        vertical_stretch_range[0], vertical_stretch_range[1]
    )
    vertical_stretch = max(0.1, vertical_stretch)

    return {
        "baseline_offset": baseline_offset,
        "char_spacing": char_spacing,
        "rotation_deg": rotation_deg,
        "vertical_stretch": vertical_stretch
    }

In [269]:
def plan_word(
    word,
    characters,
    baseline_anchor,
    base_letter_spacing,
    base_word_spacing,
    baseline_jitter=0,
    char_spacing_jitter=0,
    word_spacing_jitter=0,
    rotation_range=(0.0, 0.0),
    vertical_stretch_range=(1.0, 1.0)
):
    planned_chars = []

    for char in word:
        if char not in characters or len(characters[char]) == 0:
            continue

        raw_img = random.choice(characters[char])

        structure = sample_structure_values(
            base_letter_spacing=base_letter_spacing,
            base_word_spacing=base_word_spacing,
            baseline_jitter=baseline_jitter,
            char_spacing_jitter=char_spacing_jitter,
            word_spacing_jitter=word_spacing_jitter,
            rotation_range=rotation_range,
            vertical_stretch_range=vertical_stretch_range
        )

        transformed_img, transformed_anchor = transform_character(
            raw_img,
            baseline_anchor=baseline_anchor,
            rotation_deg=structure["rotation_deg"],
            vertical_stretch=structure["vertical_stretch"]
        )

        planned_chars.append({
            "char": char,
            "img": transformed_img,
            "anchor_y": transformed_anchor,
            "baseline_offset": structure["baseline_offset"],
            "char_spacing": structure["char_spacing"]
        })

    word_spacing = base_word_spacing + random.randint(
        -word_spacing_jitter, word_spacing_jitter
    )
    word_spacing = max(0, word_spacing)

    word_width = 0
    if planned_chars:
        for item in planned_chars:
            word_width += item["img"].size[0] + item["char_spacing"]
        word_width -= planned_chars[-1]["char_spacing"]

    return {
        "chars": planned_chars,
        "word_spacing": word_spacing,
        "word_width": word_width
    }


def generate_text_image(
    text,
    characters,
    output_path="text.png",
    canvas_width=5000,
    canvas_height=6000,
    margin=60,
    line_spacing=40,
    base_letter_spacing=6,
    base_word_spacing=30,
    baseline_jitter=0,
    char_spacing_jitter=0,
    word_spacing_jitter=0,
    rotation_range=(0.0, 0.0),
    vertical_stretch_range=(1.0, 1.0),
    seed=None
):
    if seed is not None:
        random.seed(seed)

    canvas = Image.new("L", (canvas_width, canvas_height), color=255)

    sample_img = characters["a"][0]
    image_height = sample_img.size[1]

    baseline_anchor = int(image_height * (2 / 3))

    x = margin
    line_top = margin
    current_line_bottom = line_top

    max_x_used = margin

    for word in text.split(" "):
        planned_word = plan_word(
            word=word,
            characters=characters,
            baseline_anchor=baseline_anchor,
            base_letter_spacing=base_letter_spacing,
            base_word_spacing=base_word_spacing,
            baseline_jitter=baseline_jitter,
            char_spacing_jitter=char_spacing_jitter,
            word_spacing_jitter=word_spacing_jitter,
            rotation_range=rotation_range,
            vertical_stretch_range=vertical_stretch_range
        )

        word_width = planned_word["word_width"]

        if x + word_width > canvas_width - margin and x > margin:
            x = margin
            line_top = current_line_bottom + line_spacing
            current_line_bottom = line_top

        line_baseline = line_top + baseline_anchor

        for item in planned_word["chars"]:
            img = item["img"]
            anchor_y = item["anchor_y"]
            baseline_offset = item["baseline_offset"]
            char_spacing = item["char_spacing"]

            paste_y = int(round(line_baseline + baseline_offset - anchor_y))

            canvas.paste(img, (x, paste_y))

            max_x_used = max(max_x_used, x + img.size[0])

            x += img.size[0] + char_spacing
            current_line_bottom = max(current_line_bottom, paste_y + img.size[1])

        x += planned_word["word_spacing"]

    final_height = current_line_bottom + margin
    final_width = max_x_used + margin

    canvas = canvas.crop((0, 0, final_width, final_height))

    canvas.save(output_path)
    return canvas

In [270]:
PASSAGE_A = "Maya arrived at the community garden just after sunrise, when the air was still cool and the sidewalks were quiet. She had promised her neighbor, Mrs. Alvarez, that she would water the tomato plants while she was away visiting her sister. At first, everything looked normal: the gate was closed, the hose was coiled beside the shed, and the small wooden signs still marked each row of vegetables. But when Maya walked toward the back fence, she noticed that several tomato plants had fallen sideways during the night. Their thin green stems were bent toward the soil, and a few pale tomatoes rested in the dirt. Maya felt nervous because she did not want Mrs. Alvarez to come home and think she had been careless. She searched the shed and found a roll of string, a pair of scissors, and several narrow sticks. One by one, she lifted the plants gently and tied them to the sticks without pulling too tightly. Then she watered the soil around their roots and brushed dirt from the tomatoes. By the time the sun rose above the apartment buildings, the row looked almost normal again. Maya took a picture to send to Mrs. Alvarez, but then she decided not to. She wanted the garden to look peaceful when her neighbor returned, as if nothing bad had happened at all."

PASSAGE_B = "Leo stood at the bus stop with his backpack between his feet and a library book tucked under one arm. The morning had started with bright sunlight, but a gray cloud now covered the sky, and rainwater from the night before still gathered in shallow puddles along the curb. Leo was supposed to return the book before school because it was already two days late. He had finished the final chapter at breakfast and felt proud that he had remembered to bring it with him. While he waited, he opened the book again and reread his favorite page, the one where the main character finally finds the missing map. A delivery truck splashed through a puddle, and Leo stepped backward to avoid the water. At that exact moment, the bus turned the corner. He closed the book quickly, grabbed his backpack, and hurried toward the door. Just as he reached the first step, he realized that his hands were empty. The book was still lying on the bench. Leo ran back, snatched it up, and returned just before the doors closed. He climbed onto the bus breathing hard, with rain on his shoes and mud on one sleeve. The driver smiled at him in the mirror and said, \"A careful reader should also be a careful traveler.\" Leo laughed, but he held the book tightly for the rest of the ride."

PASSAGE_C = "Nora found the blue envelope while she was helping her mother clean the kitchen shelves. They had already thrown away old grocery lists, empty jars, and a box of tea that smelled like dust instead of mint. The envelope was hidden inside a cookbook that no one had opened in years. Its paper was thin and soft, and the front was covered with handwriting Nora recognized from birthday cards: her grandmother's round, careful letters. Inside the envelope was a recipe for apple cake, folded around a short note. The note said that the cake should be made on rainy days, when the house felt too quiet and everyone needed a reason to gather in the kitchen. Nora read the sentence twice and looked out the window. Rain was tapping lightly against the glass. Her mother was about to put the recipe back in the book, but Nora asked if they could make it before dinner. They found apples in the fruit bowl, cinnamon in the cabinet, and a small pan behind the mixing bowls. As the cake baked, the kitchen filled with a warm, sweet smell. Nora's mother told stories about how her grandmother used to hum while cooking and always forgot where she had placed her glasses. By the time the cake was ready, the kitchen no longer felt like a room they were cleaning. It felt like a place that had remembered someone."

PASSAGE_D = "Ethan was cleaning his room because his cousins were coming to visit that afternoon. His mother had told him that no one could sleep on the floor if the floor could not be found. Ethan thought this was unfair, but he had to admit that books, socks, and game pieces were spread across almost every corner. He had just pushed a pile of shirts into a drawer when he heard a soft tapping sound. At first, he thought a branch was hitting the window, so he pulled back the curtain and looked outside. The tree beside the house was completely still. He heard the tapping again, this time from behind him. Ethan turned slowly and followed the sound to the closet. When he opened the door, his little sister, Lily, was crouched behind a pile of coats with a wooden spoon in her hand. She had been tapping the wall and trying not to laugh. Ethan wanted to be annoyed, especially because he still had so much cleaning left to do. Instead, he picked up a flashlight from his desk and shined it into the closet like an explorer entering a cave. Lily whispered that the coats were mountains and the laundry basket was a sleeping bear. Ethan added that the missing socks were ancient treasure. By the time their mother came upstairs, the room was only half clean, but the closet had become the entrance to a secret underground kingdom."

In [271]:
os.makedirs('generated_texts', exist_ok=True)

for i in range(4):
    if i == 0:
        PASSAGE = PASSAGE_A
    elif i == 1:
        PASSAGE = PASSAGE_B
    elif i == 2:
        PASSAGE = PASSAGE_C
    else:
        PASSAGE = PASSAGE_D
    passage_id = chr(ord('A') + i)
    for j in range(4):
        if j%2 == 0:
            character_variation = "LOW"
        else:
            character_variation = "HIGH"
        if j//2 == 0:
            structure_variation = "LOW"
        else:
            structure_variation = "HIGH"

        characters = load_character_templates(character_variation, num_variants=30)

        structure_settings = STRUCTURE_PRESETS[structure_variation]

        img = generate_text_image(
            text=PASSAGE,
            characters=characters,
            output_path=f"generated_texts/{passage_id}_{character_variation}_{structure_variation}.png",
            **structure_settings
        )
